In [6]:
import requests
import pandas as pd

# 1. Base URL for openFDA Drug Adverse Event API
url = "https://api.fda.gov/drug/event.json"

# 2. Corrected Query Parameters: Filter for Vincristine AND Pediatric patients (0 to 18 years old)
# Field path updated to 'patient.patientonsetage' to resolve the 404 error
params = {
    'search': 'patient.drug.medicinalproduct:vincristine AND patient.patientonsetage:[0 TO 18]',
    'limit': 100
}

print("⚡ Connecting to FDA servers and fetching data... Please wait...")
response = requests.get(url, params=params)

if response.status_code == 200:
    raw_data = response.json()
    results = raw_data.get('results', [])
    
    cleaned_records = []
    
    for idx, case in enumerate(results):
        # Extract basic demographics safely
        patient_info = case.get('patient', {})
        age = patient_info.get('patientonsetage', 'Unknown')
        sex = patient_info.get('patientsex', 'Unknown')
        weight = patient_info.get('patientweight', 'Unknown')
        country = case.get('reportercountry', 'Unknown')
        
        # Convert API gender codes to readable text (1 = Male, 2 = Female)
        if sex == '1': sex = 'Male'
        elif sex == '2': sex = 'Female'
        else: sex = 'Unknown'
        
        # Extract and format all reported side effects (Reactions)
        reactions_list = patient_info.get('reaction', [])
        reactions = [r.get('reactionmeddrapt', '').lower() for r in reactions_list if r.get('reactionmeddrapt')]
        reactions_str = ", ".join(reactions)
        
        # Define Target Variable (Y): Check for Vincristine-Induced Peripheral Neurotoxicity (VIPN) keywords
        neuro_keywords = ['peripheral neuropathy', 'neuropathy peripheral', 'paresthesia', 'foot drop', 'gait disturbance', 'neuralgia']
        has_vipn = 1 if any(kw in reactions_str for kw in neuro_keywords) else 0
        
        # Extract and format all concomitant medications taken by the patient
        drugs_list = patient_info.get('drug', [])
        all_drugs = [d.get('medicinalproduct', '').lower() for d in drugs_list if d.get('medicinalproduct')]
        drugs_str = ", ".join(all_drugs)
        
        # Resource-Limited Predictor: Check for concomitant Azole Antifungals (e.g., Fluconazole)
        azole_keywords = ['fluconazole', 'voriconazole', 'itraconazole', 'posaconazole']
        has_azole = 1 if any(az in drugs_str for az in azole_keywords) else 0
        
        # Append the structured variables into a clean record
        cleaned_records.append({
            'Case_ID': idx + 1,
            'Age': age,
            'Sex': sex,
            'Weight_kg': weight,
            'Country': country,
            'Concomitant_Azole': has_azole,
            'All_Side_Effects': reactions_str,
            'VIPN_Toxicity_Target': has_vipn  # Target variable for ML
        })
        
    # Convert the list of records into a Pandas DataFrame
    df = pd.DataFrame(cleaned_records)
    
    # 3. Export the processed data directly into a CSV file inside your repository
    output_filename = "vincristine_pediatric_data.csv"
    df.to_csv(output_filename, index=False)
    
    print(f"\n🎉 Success! Data extraction completed successfully.")
    print(f"📁 Output file saved as: '{output_filename}'")
    print(f"📊 Total records processed: {len(df)}")
    
    # Preview the top 5 rows
    print("\n--- Previewing a small sample of your structured dataset ---")
    print(df[['Age', 'Sex', 'Country', 'Concomitant_Azole', 'VIPN_Toxicity_Target']].head())

else:
    print(f"❌ Connection failed! openFDA server returned status code: {response.status_code}")
    print("Response text:", response.text)


⚡ Connecting to FDA servers and fetching data... Please wait...



🎉 Success! Data extraction completed successfully.
📁 Output file saved as: 'vincristine_pediatric_data.csv'
📊 Total records processed: 100

--- Previewing a small sample of your structured dataset ---
  Age     Sex  Country  Concomitant_Azole  VIPN_Toxicity_Target
0   3    Male  Unknown                  0                     0
1   1  Female  Unknown                  0                     0
2   6    Male  Unknown                  0                     0
3   2  Female  Unknown                  0                     0
4  12    Male  Unknown                  0                     0


In [10]:
import requests
import pandas as pd

# 1. Base URL for openFDA Drug Adverse Event API
url = "https://api.fda.gov/drug/event.json"

# 2. Safe Query Parameters: Capped at 100 records to avoid 403 authorization limits
params = {
    'search': 'patient.drug.medicinalproduct:vincristine AND patient.patientonsetage:[0 TO 18]',
    'limit': 100
}

print("⚡ Connecting to openFDA for safe data download (100 records)...")
response = requests.get(url, params=params)

if response.status_code == 200:
    results = response.json().get('results', [])
    cleaned_records = []
    
    for idx, case in enumerate(results):
        patient_info = case.get('patient', {})
        
        # Age Handling
        age = patient_info.get('patientonsetage')
        try:
            age = float(age) if age is not None else None
        except ValueError:
            age = None
            
        # Sex Handling
        sex = patient_info.get('patientsex', 'Unknown')
        if sex == '1': sex = 'Male'
        elif sex == '2': sex = 'Female'
        else: sex = 'Unknown'
        
        # Country Handling
        country = case.get('reportercountry', 'Unknown')
        
        # Reactions / Neurotoxicity Check
        reactions_list = patient_info.get('reaction', [])
        reactions = [r.get('reactionmeddrapt', '').lower() for r in reactions_list if r.get('reactionmeddrapt')]
        reactions_str = ", ".join(reactions)
        
        # Comprehensive Neurotoxicity Mapping (Clinical Keywords)
        neuro_keywords = [
            'peripheral neuropathy', 'neuropathy peripheral', 'paresthesia', 
            'foot drop', 'gait disturbance', 'neuralgia', 'hypoesthesia',
            'polyneuropathy', 'muscle weakness', 'areflexia', 'nerve injury'
        ]
        has_vipn = 1 if any(kw in reactions_str for kw in neuro_keywords) else 0
        
        # Concomitant Medications Check
        drugs_list = patient_info.get('drug', [])
        all_drugs = [d.get('medicinalproduct', '').lower() for d in drugs_list if d.get('medicinalproduct')]
        drugs_str = ", ".join(all_drugs)
        
        # Resource-Limited Predictor: Concomitant Azole Antifungals
        azole_keywords = ['fluconazole', 'voriconazole', 'itraconazole', 'posaconazole', 'ketoconazole']
        has_azole = 1 if any(az in drugs_str for az in azole_keywords) else 0
        
        cleaned_records.append({
            'Age_Years': age,
            'Sex': sex,
            'Country': country,
            'Concomitant_Azole': has_azole,
            'VIPN_Toxicity_Target': has_vipn
        })
        
    # Convert and Save Dataset
    df = pd.DataFrame(cleaned_records)
    df.to_csv("vincristine_pediatric_dataset.csv", index=False)
    
    print("\n🎉 Dataset successfully created!")
    print(f"📊 Total Rows Gathered: {len(df)}")
    print(f"🧠 Total Neurotoxicity (VIPN) Cases Found: {df['VIPN_Toxicity_Target'].sum()}")
    print(f"💊 Total Cases with Co-prescribed Azoles: {df['Concomitant_Azole'].sum()}")
    print("\n--- Top 5 Sample Rows ---")
    print(df.head())
    
else:
    print(f"❌ Failed to fetch dataset. Status code: {response.status_code}")


⚡ Connecting to openFDA for safe data download (100 records)...

🎉 Dataset successfully created!
📊 Total Rows Gathered: 100
🧠 Total Neurotoxicity (VIPN) Cases Found: 6
💊 Total Cases with Co-prescribed Azoles: 0

--- Top 5 Sample Rows ---
   Age_Years     Sex  Country  Concomitant_Azole  VIPN_Toxicity_Target
0        3.0    Male  Unknown                  0                     0
1        1.0  Female  Unknown                  0                     0
2        6.0    Male  Unknown                  0                     0
3        2.0  Female  Unknown                  0                     0
4       12.0    Male  Unknown                  0                     0


In [11]:
import requests
import pandas as pd
import time

url = "https://fda.gov"
cleaned_records = []
total_records_needed = 1000
records_per_page = 100

print("⚡ Starting safe batch extraction (100 records per batch) with delay...")

# Loop 10 times to safely gather 1,000 records total
for skip_val in range(0, total_records_needed, records_per_page):
    params = {
        'search': 'patient.drug.medicinalproduct:vincristine AND patient.patientonsetage:[0 TO 18]',
        'limit': records_per_page,
        'skip': skip_val
    }
    
    print(f"   → Fetching records {skip_val} to {skip_val + records_per_page}...")
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        results = response.json().get('results', [])
        
        for case in results:
            patient_info = case.get('patient', {})
            
            # Extract and process Age
            age = patient_info.get('patientonsetage')
            try:
                age = float(age) if age is not None else None
            except ValueError:
                age = None
                
            # Extract and process Sex
            sex = patient_info.get('patientsex', 'Unknown')
            if sex == '1': sex = 'Male'
            elif sex == '2': sex = 'Female'
            else: sex = 'Unknown'
            
            country = case.get('reportercountry', 'Unknown')
            
            # Neurotoxicity Check
            reactions_list = patient_info.get('reaction', [])
            reactions = [r.get('reactionmeddrapt', '').lower() for r in reactions_list if r.get('reactionmeddrapt')]
            reactions_str = ", ".join(reactions)
            
            neuro_keywords = [
                'peripheral neuropathy', 'neuropathy peripheral', 'paresthesia', 
                'foot drop', 'gait disturbance', 'neuralgia', 'hypoesthesia',
                'polyneuropathy', 'muscle weakness', 'areflexia', 'nerve injury'
            ]
            has_vipn = 1 if any(kw in reactions_str for kw in neuro_keywords) else 0
            
            # Concomitant Medications / Azoles Check
            drugs_list = patient_info.get('drug', [])
            all_drugs = [d.get('medicinalproduct', '').lower() for d in drugs_list if d.get('medicinalproduct')]
            drugs_str = ", ".join(all_drugs)
            
            azole_keywords = ['fluconazole', 'voriconazole', 'itraconazole', 'posaconazole', 'ketoconazole']
            has_azole = 1 if any(az in drugs_str for az in azole_keywords) else 0
            
            cleaned_records.append({
                'Age_Years': age,
                'Sex': sex,
                'Country': country,
                'Concomitant_Azole': has_azole,
                'VIPN_Toxicity_Target': has_vipn
            })
            
        # Crucial 2-second delay to avoid triggering FDA security blocks
        time.sleep(2)
    else:
        print(f"⚠️ FDA server requested a pause at batch {skip_val}. Status code: {response.status_code}")
        print("Waiting 5 seconds before trying the next batch...")
        time.sleep(5)

# Convert and save the full 1,000-record dataset
df = pd.DataFrame(cleaned_records)
df.to_csv("vincristine_pediatric_large_dataset.csv", index=False)

print("\n🎉 Publication-grade dataset successfully built!")
print(f"📊 Total Rows Gathered: {len(df)}")
print(f"🧠 Total Neurotoxicity (VIPN) Cases Found: {df['VIPN_Toxicity_Target'].sum()}")
print(f"💊 Total Cases with Co-prescribed Azoles: {df['Concomitant_Azole'].sum()}")


⚡ Starting safe batch extraction (100 records per batch) with delay...
   → Fetching records 0 to 100...
⚠️ FDA server requested a pause at batch 0. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 100 to 200...
⚠️ FDA server requested a pause at batch 100. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 200 to 300...
⚠️ FDA server requested a pause at batch 200. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 300 to 400...
⚠️ FDA server requested a pause at batch 300. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 400 to 500...
⚠️ FDA server requested a pause at batch 400. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 500 to 600...
⚠️ FDA server requested a pause at batch 500. Status code: 404
Waiting 5 seconds before trying the next batch...
   → Fetching records 600 to 700...
⚠️ 

KeyError: 'VIPN_Toxicity_Target'

In [2]:
!pip install requests pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 24.6 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 35.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [12]:
import requests
import pandas as pd
import time

url = "https://api.fda.gov/drug/event.json"
cleaned_records = []

# Breaking down by pediatric age brackets to bypass pagination locks
age_brackets = [
    "[0 TO 4]",
    "[5 TO 9]",
    "[10 TO 14]",
    "[15 TO 18]"
]

print("⚡ Starting bracketed data extraction across pediatric groups...")

for bracket in age_brackets:
    params = {
        'search': f'patient.drug.medicinalproduct:vincristine AND patient.patientonsetage:{bracket}',
        'limit': 100  # Pulling a safe max capacity of 100 per distinct group
    }
    
    print(f"   → Fetching pediatric age range: {bracket}...")
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        results = response.json().get('results', [])
        print(f"     ✅ Successfully retrieved {len(results)} records.")
        
        for case in results:
            patient_info = case.get('patient', {})
            
            # Process Age
            age = patient_info.get('patientonsetage')
            try:
                age = float(age) if age is not None else None
            except ValueError:
                age = None
                
            # Process Sex
            sex = patient_info.get('patientsex', 'Unknown')
            if sex == '1': sex = 'Male'
            elif sex == '2': sex = 'Female'
            else: sex = 'Unknown'
            
            country = case.get('reportercountry', 'Unknown')
            
            # Neurotoxicity Mapping
            reactions_list = patient_info.get('reaction', [])
            reactions = [r.get('reactionmeddrapt', '').lower() for r in reactions_list if r.get('reactionmeddrapt')]
            reactions_str = ", ".join(reactions)
            
            neuro_keywords = [
                'peripheral neuropathy', 'neuropathy peripheral', 'paresthesia', 
                'foot drop', 'gait disturbance', 'neuralgia', 'hypoesthesia',
                'polyneuropathy', 'muscle weakness', 'areflexia', 'nerve injury'
            ]
            has_vipn = 1 if any(kw in reactions_str for kw in neuro_keywords) else 0
            
            # Concomitant Medications Check
            drugs_list = patient_info.get('drug', [])
            all_drugs = [d.get('medicinalproduct', '').lower() for d in drugs_list if d.get('medicinalproduct')]
            drugs_str = ", ".join(all_drugs)
            
            azole_keywords = ['fluconazole', 'voriconazole', 'itraconazole', 'posaconazole', 'ketoconazole']
            has_azole = 1 if any(az in drugs_str for az in azole_keywords) else 0
            
            cleaned_records.append({
                'Age_Years': age,
                'Sex': sex,
                'Country': country,
                'Concomitant_Azole': has_azole,
                'VIPN_Toxicity_Target': has_vipn
            })
        
        # Safe delay
        time.sleep(2)
    else:
        print(f"❌ Failed for bracket {bracket}. Status code: {response.status_code}")

# Convert into our primary workspace DataFrame
df = pd.DataFrame(cleaned_records)
df.to_csv("vincristine_pediatric_large_dataset.csv", index=False)

print("\n🎉 Multi-bracket dataset successfully built!")
print(f"📊 Total Rows Gathered: {len(df)}")
print(f"🧠 Total Neurotoxicity (VIPN) Cases Found: {df['VIPN_Toxicity_Target'].sum()}")
print(f"💊 Total Cases with Co-prescribed Azoles: {df['Concomitant_Azole'].sum()}")


⚡ Starting bracketed data extraction across pediatric groups...
   → Fetching pediatric age range: [0 TO 4]...
     ✅ Successfully retrieved 100 records.
   → Fetching pediatric age range: [5 TO 9]...
     ✅ Successfully retrieved 100 records.
   → Fetching pediatric age range: [10 TO 14]...
     ✅ Successfully retrieved 100 records.
   → Fetching pediatric age range: [15 TO 18]...
     ✅ Successfully retrieved 100 records.

🎉 Multi-bracket dataset successfully built!
📊 Total Rows Gathered: 400
🧠 Total Neurotoxicity (VIPN) Cases Found: 20
💊 Total Cases with Co-prescribed Azoles: 10


In [13]:
!pip install scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 40.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset we built
df = pd.read_csv("vincristine_pediatric_large_dataset.csv")

# 2. Data Cleaning: Drop rows where Age is missing
df = df.dropna(subset=['Age_Years'])

# 3. Feature Engineering: Convert 'Sex' text to numbers (One-Hot Encoding)
df = pd.get_dummies(df, columns=['Sex'], drop_first=True)

# 4. Separate Features (X) and Target (Y)
feature_cols = ['Age_Years', 'Concomitant_Azole'] + [col for col in df.columns if 'Sex_' in col]
X = df[feature_cols]
Y = df['VIPN_Toxicity_Target']

# 5. Corrected Train-Test Split (Using test_size instead of test_split)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)

print("🎉 Preprocessing and Dataset Split Complete!")
print(f"📈 Training Set Size: {X_train.shape[0]} patients")
print(f"📉 Testing Set Size: {X_test.shape[0]} patients")
print("\n--- Features the model will look at ---")
print(X_train.head())


🎉 Preprocessing and Dataset Split Complete!
📈 Training Set Size: 320 patients
📉 Testing Set Size: 80 patients

--- Features the model will look at ---
     Age_Years  Concomitant_Azole  Sex_Male  Sex_Unknown
91         3.0                  0     False        False
8          1.0                  0     False        False
330       15.0                  0      True        False
347       18.0                  0      True        False
189        9.0                  0     False        False


In [16]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# 1. Initialize and Train the Logistic Regression Model
# We use class_weight='balanced' because toxicity cases are rare (imbalanced data)
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, Y_train)

# 2. Make Predictions on the Test Dataset
Y_pred = model.predict(X_test)
Y_prob = model.predict_proba(X_test)[:, 1] # Probability scores for AUC-ROC calculation

# 3. Calculate Clinical Metrics for the Paper
accuracy = accuracy_score(Y_test, Y_pred)
auc_roc = roc_auc_score(Y_test, Y_prob)
class_report = classification_report(Y_test, Y_pred)

print("🎉 Machine Learning Model Training Complete!")
print("==============================================")
print(f"📊 Overall Model Accuracy: {accuracy * 100:.2f}%")
print(f"🧠 Clinical AUC-ROC Score: {auc_roc:.3f} (Closer to 1.0 means highly accurate prediction)")
print("==============================================")
print("\n📝 Detailed Performance Report (Precision, Recall/Sensitivity):")
print(class_report)

print("==============================================")
print("🩺 CLINICAL INSIGHTS (Odds Ratios for your Manuscript):")
# Calculate Odds Ratios by exponentiating the model coefficients
odds_ratios = np.exp(model.coef_[0])
for feature, or_value in zip(X_train.columns, odds_ratios):
    print(f"   → {feature}: Odds Ratio = {or_value:.3f}")
print("==============================================")


🎉 Machine Learning Model Training Complete!
📊 Overall Model Accuracy: 52.50%
🧠 Clinical AUC-ROC Score: 0.691 (Closer to 1.0 means highly accurate prediction)

📝 Detailed Performance Report (Precision, Recall/Sensitivity):
              precision    recall  f1-score   support

           0       0.97      0.51      0.67        76
           1       0.07      0.75      0.14         4

    accuracy                           0.53        80
   macro avg       0.53      0.63      0.40        80
weighted avg       0.93      0.53      0.65        80

🩺 CLINICAL INSIGHTS (Odds Ratios for your Manuscript):
   → Age_Years: Odds Ratio = 1.081
   → Concomitant_Azole: Odds Ratio = 0.336
   → Sex_Male: Odds Ratio = 0.990
   → Sex_Unknown: Odds Ratio = 0.850
